# Train Freiburg Mobility Aids YOLOv8 on Google Colab

Run this notebook on a Colab GPU runtime to prepare the Freiburg RGB dataset, train YOLOv8 Nano for 30 epochs, and download `weights/best.pt` to your computer.

> **Before you start:** In Colab choose **Runtime → Change runtime type → T4 GPU** (if available), then use **Runtime → Run all**. Free GPU availability varies by account and time; code cannot reserve a T4. The image archive is 12.4 GB and extraction plus prepared copies need about 34.6 GiB free disk space. Keep the runtime connected until training and download finish.

The notebook stores data and model files in the temporary Colab VM under `/content/Gut-ankommen`; they disappear when the runtime is deleted unless you download or copy the model to Drive.

## 1. Check runtime and available disk space

This cell confirms a CUDA GPU is attached and prints its name. It warns if the selected GPU is not a T4.

In [ ]:
import shutil
import subprocess

try:
    import torch
except ImportError as exc:
    raise RuntimeError("PyTorch is not available in this Colab runtime. Select a GPU runtime and restart it.") from exc

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU is active. In Colab choose Runtime → Change runtime type → T4 GPU, save, then rerun this notebook.")

gpu_name = torch.cuda.get_device_name(0)
print(f"GPU: {gpu_name}")
if "T4" not in gpu_name.upper():
    print("Note: this runtime is using a different CUDA GPU than a T4. Continuing with the available GPU.")

free_gib = shutil.disk_usage("/content").free / (1024 ** 3)
print(f"Free Colab disk: {free_gib:.1f} GiB")
if free_gib < 36:
    raise RuntimeError("This runtime has less than 36 GiB free. Freiburg RGB preparation needs about 34.6 GiB for the archive, extraction, and prepared copy. Restart with a runtime that has more disk or use a larger Colab plan.")

## 2. Clone the project repository

The notebook clones the current `main` branch into the Colab VM. If the project is already present, it updates the checkout.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/MarcZem/Gut-ankommen.git"
REPO_DIR = Path("/content/Gut-ankommen")
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

%cd /content/Gut-ankommen
print("Project files:", sorted(p.name for p in REPO_DIR.iterdir() if p.is_file()))

## 3. Install training and dataset dependencies

Installs the requested Ultralytics, OpenCV, and tqdm packages, plus YAML and image libraries used by the Freiburg converter.

In [ ]:
%pip install -q ultralytics opencv-python tqdm PyYAML Pillow

import torch
from ultralytics import __version__ as ultralytics_version
print(f"Ultralytics {ultralytics_version}; CUDA available: {torch.cuda.is_available()}")

## 4. Download and prepare the Freiburg RGB dataset

Downloads the official `Images_RGB.zip` and `Annotations_RGB.zip` archives directly to Colab storage, extracts them, converts Freiburg YAML boxes to normalized YOLO labels, and creates the train/validation split and `data.yaml`. The image archive is large; avoid disconnecting the runtime during this cell.

In [ ]:
import subprocess

subprocess.run(["python", "dataset_prep.py"], check=True)
data_yaml = Path("data/freiburg_yolo/data.yaml")
if not data_yaml.is_file():
    raise FileNotFoundError(f"Dataset config was not generated: {data_yaml}")
print(data_yaml.read_text())

## 5. Train YOLOv8 Nano for 30 epochs

`train_yolo.py` selects CUDA automatically in Colab. Training uses 640px images and copies the best checkpoint to `weights/best.pt`. Keep the notebook session active until this cell finishes.

In [ ]:
subprocess.run(["python", "train_yolo.py", "--epochs", "30", "--imgsz", "640"], check=True)

weights_path = Path("weights/best.pt")
if not weights_path.is_file():
    raise FileNotFoundError(f"Training completed without the expected checkpoint: {weights_path}")
print(f"Ready to download: {weights_path.resolve()} ({weights_path.stat().st_size / (1024 ** 2):.1f} MiB)")

## 6. Download `best.pt` to your computer

Run this cell after training. Colab will start a browser download. Save the file into your local repository's `weights/` folder so `python webcam_demo.py` can load it from `./weights/best.pt`.

In [ ]:
from google.colab import files
files.download(str(weights_path))

## Optional: save a copy to Google Drive

Use this instead of, or in addition to, the browser download if you want to keep a persistent cloud copy.

In [ ]:
from google.colab import drive
import shutil

drive.mount("/content/drive")
drive_weights = Path("/content/drive/MyDrive/Gut-ankommen/weights")
drive_weights.mkdir(parents=True, exist_ok=True)
shutil.copy2(weights_path, drive_weights / "best.pt")
print(f"Saved to {drive_weights / 'best.pt'}")

## Run the trained model locally

Copy the downloaded file to `weights/best.pt` inside your local clone, install the dependencies from `requirements.txt`, and run:

```bash
python webcam_demo.py --weights ./weights/best.pt
```

The webcam demo draws red boxes, class names, and confidence scores around detections. The notebook does not upload your camera stream or model weights to Vercel.